In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Upload the CSV files provided by Student 1
uploaded = files.upload()

Saving X_test_scaled.csv to X_test_scaled.csv
Saving y_test.csv to y_test.csv
Saving y_train.csv to y_train.csv
Saving X_train_scaled.csv to X_train_scaled.csv


In [ ]:
X_train = pd.read_csv('X_train_scaled.csv')
X_test = pd.read_csv('X_test_scaled.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (455, 30), X_test shape: (114, 30)


In [ ]:
!pip install scikit-opt

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sko.GA import GA

In [ ]:
def fitness_function(x):
    # x is a binary vector (1s and 0s) representing features
    selected_features = np.where(x == 1)[0]

    # Penalty if the GA tries to select 0 features
    if len(selected_features) == 0:
        return 0.0

    # Subset training and testing data to only use the selected features
    xtrain_sub = X_train.iloc[:, selected_features]
    xtest_sub = X_test.iloc[:, selected_features]

    # Train a quick classifier (e.g., KNN)
    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(xtrain_sub, y_train)
    preds = clf.predict(xtest_sub)

    # Calculate accuracy
    acc = accuracy_score(y_test, preds)

    # Fitness formula: Weight accuracy heavily, but reward feature reduction
    # Minimize feature ratio while maximizing accuracy
    feature_penalty = 0.01 * (len(selected_features) / X_train.shape[1])
    fitness = acc - feature_penalty

    return fitness

In [ ]:
def objective_function(x):
    selected_features = np.where(np.array(x) == 1)[0]

    if len(selected_features) == 0:
        return 1.0 # Worst penalty if no features are chosen

    xtrain_sub = X_train.iloc[:, selected_features]
    xtest_sub = X_test.iloc[:, selected_features]

    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(xtrain_sub, y_train)
    preds = clf.predict(xtest_sub)

    error = 1.0 - accuracy_score(y_test, preds)
    feature_ratio = len(selected_features) / X_train.shape[1]

    # Objective: Minimize error and minimize the fraction of features used
    alpha = 0.99 # weight for accuracy error
    beta = 0.01  # weight for feature reduction ratio

    cost = (alpha * error) + (beta * feature_ratio)
    return cost

In [ ]:
# Number of features is 30
n_features = X_train.shape[1]

# Initialize GA using scikit-opt
# func=objective_function, n_dim=30 features, size_pop=40, max_iter=30, prob_mut=0.001
ga = GA(func=objective_function,
        n_dim=n_features,
        size_pop=40,
        max_iter=30,
        lb=[0]*n_features,
        ub=[1]*n_features,
        precision=1) # precision=1 forces binary (0 or 1)

# Run optimization
best_x, best_y = ga.run()

print("Optimization complete!")
print(f"Best feature vector (1=keep, 0=discard): {best_x}")

Optimization complete!
Best feature vector (1=keep, 0=discard): [0. 1. 0. 0. 0. 0. 1. 1. 0. 1. 0. 1. 1. 1. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 1. 1. 0. 0.]


In [ ]:
# Get indices of features selected by GA
ga_selected_indices = np.where(np.array(best_x) == 1)[0]
ga_selected_feature_names = X_train.columns[ga_selected_indices]

print(f"Total features selected by GA: {len(ga_selected_indices)} out of {n_features}")
print(f"Selected feature names: list({ga_selected_feature_names})")

# Train final model with GA features
X_train_ga = X_train.iloc[:, ga_selected_indices]
X_test_ga = X_test.iloc[:, ga_selected_indices]

final_clf = KNeighborsClassifier(n_neighbors=5)
final_clf.fit(X_train_ga, y_train)
ga_preds = final_clf.predict(X_test_ga)

# Calculate final GA metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("\n--- GA MODEL PERFORMANCE ---")
print(f"Accuracy:  {accuracy_score(y_test, ga_preds):.4f}")
print(f"Precision: {precision_score(y_test, ga_preds):.4f}")
print(f"Recall:    {recall_score(y_test, ga_preds):.4f}")
print(f"F1-Score:  {f1_score(y_test, ga_preds):.4f}")

Total features selected by GA: 11 out of 30
Selected feature names: list(Index(['x.texture_mean', 'x.concavity_mean', 'x.concave_pts_mean',
       'x.fractal_dim_mean', 'x.texture_se', 'x.perimeter_se', 'x.area_se',
       'x.compactness_se', 'x.symmetry_se', 'x.concavity_worst',
       'x.concave_pts_worst'],
      dtype='object'))

--- GA MODEL PERFORMANCE ---
Accuracy:  0.9825
Precision: 1.0000
Recall:    0.9535
F1-Score:  0.9762


In [ ]:
ga_results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Feature Count'],
    'GA_Score': [
        accuracy_score(y_test, ga_preds),
        precision_score(y_test, ga_preds),
        recall_score(y_test, ga_preds),
        f1_score(y_test, ga_preds),
        len(ga_selected_indices)
    ]
})
ga_results.to_csv('ga_performance_results.csv', index=False)
files.download('ga_performance_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>